# 三步訓練與診斷

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/07-training/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.6.1'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

### 可選：160 步合成圖與 held-out 評估

下面的主例只更新三步。跑完後另開一個 code cell：

```python
!python -m miniyolo.train --steps 160 --samples 32 --device cpu
from IPython.display import display
from PIL import Image
display(Image.open('artifacts/runs/grid-learning/loss.png'))
```

checkpoint.pt、history.json、validation 圖與完整指標 report.json 都存在 artifacts/runs/grid-learning/。

In [1]:
import math
import torch
from miniyolo.data import ShapeDataset, collate
from miniyolo.models import GridDetector
from miniyolo.targets import build_targets
from miniyolo.losses import grid_loss


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    dataset = ShapeDataset(n=4, size=64, seed=7, max_objects=1, allow_empty=False)
    images, anns = collate([dataset[i] for i in range(4)])
    target = build_targets(anns, 4, 64, 2)
    model = GridDetector(num_classes=2, grid_size=4, width=8)
    optimizer = torch.optim.Adam(model.parameters(), lr=.01)
    before = next(model.parameters()).detach().clone()
    model.train()
    for step in range(3):
        optimizer.zero_grad(set_to_none=True)
        prediction = model(images)
        assert prediction.shape == (4, 4, 4, 7)
        parts = grid_loss(prediction, target)
        assert torch.isfinite(parts['total'])
        parts['total'].backward()
        assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
        grad = sum(p.grad.abs().sum().item() for p in model.parameters() if p.grad is not None)
        assert math.isfinite(grad) and grad > 0
        optimizer.step()
        with torch.no_grad():
            obj = prediction[..., 4].sigmoid()
            pos_mean = obj[target['positive']].mean().item()
            neg_mean = obj[~target['positive']].mean().item()
        print('step', step, {k: round(v.item(), 4) for k, v in parts.items()},
              'positive', round(pos_mean, 4), 'negative', round(neg_mean, 4))
    assert not torch.equal(before, next(model.parameters()).detach())
    assert target['positive'].sum() == 4
    print('3 real CPU optimizer steps; parameters changed; no generalization claim')


if __name__ == '__main__':
    main()


step 0 {'total': 0.9817, 'box': 0.009, 'objectness': 0.2525, 'classification': 0.6842} positive 0.1202 negative 0.1202
step 1 {'total': 0.9248, 'box': 0.0091, 'objectness': 0.2508, 'classification': 0.6283} positive 0.119 negative 0.1181
step 2 {'total': 0.8323, 'box': 0.0094, 'objectness': 0.2491, 'classification': 0.5362} positive 0.12 negative 0.1169
3 real CPU optimizer steps; parameters changed; no generalization claim
